In [0]:
import pandas as pd
import requests
from io import StringIO

# Leer datos desde GitHub
url = "https://raw.githubusercontent.com/lissetdborjac-maker/DataAnaliticsUQ202602/refs/heads/main/caso_practico_ventas.csv"
response = requests.get(url)
response.encoding='utf-8'
df = pd.read_csv(StringIO(response.text))
print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
display(df.head(10))


In [0]:
# Últimas 5 filas
print("Últimas 5 filas:")
display(df.tail(5))

In [0]:
#FILAS TOTALES
len(df)


In [0]:
#TOTAL DE COLUMNAS
len(df.columns)

In [0]:
#PARA SABER NULOS
df.isna().sum()
#Cantidad 5 valores nulos
#precio_unitario 6 valores nulos
#satisfaccion_ cliente 2 valores nulo


In [0]:
nulos= df.isna().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)
print(porcentaje_nulos)

 _NOTA _
 
 PARA PODER INTERPRETAR ES ASÍ:
- cantidad: 5 nulos de 123 filas

5/123 ×100≈ 4.07 %

el 4.07% de las transacciones no se registró la cantidad.


In [0]:
#para saber tipos de datos
df.dtypes
#object → texto (cadenas).
#float64 → números decimales.

#Coinciden con lo esperado (numéricos para cantidad, precio_unitario, satisfaccion_cliente; texto para las demás

In [0]:
#Estadistica descriptiva
print("Cantidad:")
print("Media:", df["cantidad"].mean())
print("Mediana:", df["cantidad"].median())
print("Desviación estándar:", df["cantidad"].std())
print("Mínimo:", df["cantidad"].min())
print("Máximo:", df["cantidad"].max())

print("\nPrecio unitario:")
print("Media:", df["precio_unitario"].mean())
print("Mediana:", df["precio_unitario"].median())
print("Desviación estándar:", df["precio_unitario"].std())
print("Mínimo:", df["precio_unitario"].min())
print("Máximo:", df["precio_unitario"].max())

In [0]:
#Forma mas corta
print("Cantidad:")
print(df["cantidad"].describe())

print("\nPrecio unitario:")
print(df["precio_unitario"].describe())

#Como referencia adicional, la función describe() también proporciona los cuartiles 25%, 50% (mediana) y 75%.

In [0]:
#ANALISIS EXTRA PARA ENTENDER LOS RESULTADOS

# Media y mediana de cantidad
#Si la media/mediana de cantidad es 2 → es decir que regularmente compran 2 unidades por transacción.

#Eso ayuda a:

#Planear inventario (saber cuántas unidades se mueven en promedio).

#Diseñar promociones tipo “lleva 3 y paga 2”, sabiendo que el cliente promedio compra cerca de 2.

#Si la desviación es pequeña → la mayoría de clientes compra cantidades parecidas (poca variabilidad).

#Si es grande → hay clientes que compran muy poco y otros que compran mucho.


In [0]:
print("Dimensiones antes:", df.shape)
duplicados_completos = df.duplicated().sum()
print("Filas completamente duplicadas:", duplicados_completos)
duplicados_id = df["id_transaccion"].duplicated().sum()
print("id_transaccion repetidos:", duplicados_id)
if duplicados_id > 0:
    ids_repetidos = df.loc[df["id_transaccion"].duplicated(keep=False), "id_transaccion"].unique()
    print("IDs repetidos (primeros 10):", ids_repetidos[:10])

In [0]:
#Eliminar las 3 filas completamente duplicadas.
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["id_transaccion"], keep="first")

In [0]:
print("Dimensiones después de limpiar:", df.shape)
print("Filas completamente duplicadas después:", df.duplicated().sum())
print("id_transaccion repetidos después:", df["id_transaccion"].duplicated().sum())

In [0]:
df["metodo_pago"] = df["metodo_pago"].replace(
    {
        "tarjeta crédito": "Tarjeta de Crédito",
        "Tarjeta crédito": "Tarjeta de Crédito",
        "tarjeta Crédito": "Tarjeta de Crédito",
        "Tarjeta Crédito": "Tarjeta de Crédito"
    }
)

In [0]:
# Calcular la mediana de cantidad
mediana_cantidad = df["cantidad"].median()

# Reemplazar nulos por esa mediana
df["cantidad"] = df["cantidad"].fillna(mediana_cantidad)

In [0]:
# Calcular la moda de satisfaccion_cliente
moda_satisfaccion = df["satisfaccion_cliente"].mode()[0]

# Reemplazar nulos por esa moda
df["satisfaccion_cliente"] = df["satisfaccion_cliente"].fillna(moda_satisfaccion)

In [0]:
print("Nulos por columna:")
print(df.isna().sum())

#Se reemplazan por la mediana porque cantidad es numérica y la mediana es más robusta ante valores extremos.
#Se reemplazan por la moda porque es el nivel más frecuente y es fácil de interpretar en una escala de satisfacción.

In [0]:
# Ver la fila con precio 9999
df[df["precio_unitario"] == 9999.00]

In [0]:
#Para ropa, 9999 es atípico y muy probablemente un error de digitación.

#Por eso, en este contexto sí tiene sentido eliminar la fila

#No es coherente con la categoría (Ropa).

#Distorsionaría cualquier análisis de precios (media, desviación, etc.)

In [0]:
# Eliminar la fila con precio 9999 (outlier en Ropa)
df = df[df["precio_unitario"] != 9999.00]

In [0]:
print("Dimensiones después de eliminar el outlier:", df.shape)
print("¿Queda algún 9999?", (df["precio_unitario"] == 9999.00).sum())

In [0]:
#Nota: LO HICE POR CURIOSIDAD
#Los nulos en cantidad se llenaron con la mediana (2.0) y los de satisfaccion_cliente con la moda (4.0). Esto se hace para no perder filas y usar valores representativos de cada columna.

# Mediana de cantidad
mediana_cantidad = df["cantidad"].median()
print("Mediana de cantidad:", mediana_cantidad)

# Moda de satisfaccion_cliente
moda_satisfaccion = df["satisfaccion_cliente"].mode()[0]
print("Moda de satisfaccion_cliente:", moda_satisfaccion)

# Opcional: ver distribución de satisfaccion_cliente
print("\nFrecuencia de satisfaccion_cliente:")
print(df["satisfaccion_cliente"].value_counts().sort_index())

In [0]:
#dia_semana: sirve para analizar patrones por día (lunes vs viernes, semana vs fin de semana).

#mes: sirve para analizar patrones por mes (estacionalidad, meses fuertes/débiles).

# 1. Crear columna monto_total

df["monto_total"] = df["cantidad"] * df["precio_unitario"]
# 2. Convertir fecha a tipo datetime
df["fecha"] = pd.to_datetime(df["fecha"])

# 3. Extraer día de la semana y mes
df["dia_semana"] = df["fecha"].dt.day_name().map({
    "Monday": "Lunes",
    "Tuesday": "Martes",
    "Wednesday": "Miércoles",
    "Thursday": "Jueves",
    "Friday": "Viernes",
    "Saturday": "Sábado",
    "Sunday": "Domingo"})

df["mes"] = df["fecha"].dt.month_name().map({
    "January": "Enero",
    "February": "Febrero",
    "March": "Marzo",
    "April": "Abril",
    "May": "Mayo",
    "June": "Junio",
    "July": "Julio",
    "August": "Agosto",
    "September": "Septiembre",
    "October": "Octubre",
    "November": "Noviembre",
    "December": "Diciembre"
})

# 4. Discretización (Binning) de monto_total en Bajo, Medio, Alto
bins = [0, 100, 500, float("inf")]
labels = ["Bajo", "Medio", "Alto"]

df["rango_monto"] = pd.cut(
    df["monto_total"],
    bins=bins,
    labels=labels,
    right=False
)

In [0]:
# Ver columnas nuevas
print(df[["cantidad", "precio_unitario", "monto_total"]].head(10))

print("\nDía y mes:")
print(df[["fecha", "dia_semana", "mes"]].head(10))

print("\nRango de monto:")
print(df[["monto_total", "rango_monto"]].head(15))

print("\nFrecuencia por rango de monto:")
print(df["rango_monto"].value_counts().sort_index())

In [0]:
resumen_categoria = (
    df.groupby("categoria_producto")
    .agg(
        total_ingresos=("monto_total", "sum"),
        promedio_unidades=("cantidad", "mean")
    )
    .reset_index()
)

print(resumen_categoria)

In [0]:
tabla_pivot = pd.pivot_table(
    df,
    index="region",              # filas
    columns="categoria_producto",# columnas
    values="monto_total",        # qué sumar
    aggfunc="sum"                # función: suma
)

print(tabla_pivot)

In [0]:
satisfaccion_por_metodo = (
    df.groupby("metodo_pago")["satisfaccion_cliente"]
    .mean()
    .reset_index()
)

print(satisfaccion_por_metodo)

In [0]:
#ventas de la región Norte con una satisfacción igual o superior a 4.

df_filtrado = df[
    (df["region"] == "Norte") &
    (df["satisfaccion_cliente"] >= 4)
]

print("Filas originales:", len(df))
print("Filas después del filtro:", len(df_filtrado))

# Ver algunas filas
print(df_filtrado.head(10))

In [0]:
#muestra aleatoria equivalente al 10% de los registros conservando la proporción por region

muestra = df.groupby("region", group_keys=False).apply(
    lambda x: x.sample(frac=0.10, random_state=42)
)

print("Filas originales:", len(df))
print("Filas en la muestra:", len(muestra))

# Ver proporción de regiones antes y después
print("\nProporción de regiones en el DataFrame original:")
print(df["region"].value_counts(normalize=True))

print("\nProporción de regiones en la muestra:")
print(muestra["region"].value_counts(normalize=True))